In [50]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import f_regression, RFE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
file_path = r"C:/Users/Dalbir/Downloads/Trends-Forecasting-Analytics-MLOps-Vertex-AI/data/processed_file/sales_featured.parquet"
df = pd.read_parquet(file_path)

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185686 entries, 0 to 185685
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   order_id          185686 non-null  string        
 1   product           185686 non-null  string        
 2   quantity_ordered  185686 non-null  int64         
 3   price_each        185686 non-null  float64       
 4   order_date        185686 non-null  datetime64[ns]
 5   purchase_address  185686 non-null  string        
 6   month             185686 non-null  int64         
 7   sales             185686 non-null  float64       
 8   city              185686 non-null  category      
 9   hour              185686 non-null  int64         
 10  year              185686 non-null  UInt32        
 11  week              185686 non-null  UInt32        
 12  day               185686 non-null  UInt32        
 13  day_name          185686 non-null  string        
 14  week

In [52]:
categorical_cols = ['city', 'product', 'weekday_weekend']
drop_cols = ['order_id', 'order_date', 'sales', 'purchase_address', 'day_name', 'order_size']
numeric_cols = [col for col in df.columns if col not in categorical_cols + drop_cols]

X_num = df[numeric_cols]
y = df['sales']
df['product_mean_encoded'] = np.nan
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(df):
    train, val = df.iloc[train_idx], df.iloc[val_idx]
    means = train.groupby('product')['sales'].mean()
    df.loc[val_idx, 'product_mean_encoded'] = val['product'].map(means)

global_mean = df['sales'].mean()
df['product_mean_encoded'].fillna(global_mean, inplace=True)
X_cat = pd.get_dummies(df[['city', 'weekday_weekend']], drop_first=True)
X_encoded = pd.concat([X_num, df[['product_mean_encoded']], X_cat], axis=1)

print(X_encoded.head())


   quantity_ordered  price_each  month  hour  year  week  day  quarter  \
0                 1     1700.00     12     0  2020     1    1        4   
1                 1      600.00     12     7  2019    52    7        4   
2                 1       11.95     12    18  2019    50    4        4   
3                 1      149.99     12    15  2019    51    7        4   
4                 1       11.95     12    12  2019    51    3        4   

        SMA_3    SMA_5  ...  product_mean_encoded  city_ Austin  city_ Boston  \
0    0.000000    0.000  ...           1700.903534         False         False   
1    0.000000    0.000  ...            600.000000         False         False   
2  770.650000    0.000  ...             13.051909         False         False   
3  253.980000    0.000  ...            150.857998         False         False   
4   57.963333  494.778  ...             13.088714         False         False   

   city_ Dallas  city_ Los Angeles  city_ New York City  city_ Portl

C:\Users\Dalbir\AppData\Local\Temp\ipykernel_5304\2735547900.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['product_mean_encoded'].fillna(global_mean, inplace=True)


## Correlation Based Feature Selection (Numeric)

In [57]:
corr = X_encoded.corrwith(y).sort_values(ascending=False)
print(corr[:15])

price_each                 0.999202
product_mean_encoded       0.999202
SMA_3                      0.575449
SMA_5                      0.446030
CMA                        0.010487
city_ New York City        0.002373
hour                       0.001683
city_ Dallas               0.001127
city_ Seattle              0.000846
city_ Portland             0.000559
weekday_weekend_Weekend   -0.000165
Lag_3                     -0.000265
day                       -0.000537
year                      -0.001006
Lag_5                     -0.001271
dtype: float64


## F-regression (ANOVA F-test)

In [59]:
f_vals, p_vals = f_regression(X_encoded, y)
f_reg = pd.Series(f_vals, index=X_encoded.columns).sort_values(ascending=False)
print(f_reg.head(15))

price_each              1.162506e+08
product_mean_encoded    1.162411e+08
SMA_3                   9.192907e+04
SMA_5                   4.611464e+04
quantity_ordered        3.688607e+03
CMA                     2.042401e+01
quarter                 2.252005e+00
month                   2.214684e+00
week                    1.915081e+00
city_ New York City     1.045959e+00
city_ Boston            6.300086e-01
city_ Los Angeles       5.553050e-01
hour                    5.260237e-01
city_ San Francisco     3.266759e-01
city_ Austin            3.002404e-01
dtype: float64


## Recursive Feature Elimination (model-driven)

In [61]:
model = LinearRegression()
rfe = RFE(model, n_features_to_select=30)
rfe.fit(X_encoded, y)
rfe_features = pd.Series(rfe.support_, index=X_encoded.columns)
selected_features = rfe_features[rfe_features == True].index.tolist()
print(selected_features)

['quantity_ordered', 'price_each', 'month', 'hour', 'year', 'week', 'day', 'quarter', 'SMA_3', 'SMA_5', 'Lag_3', 'Lag_5', 'CMA', 'product_mean_encoded', 'city_ Austin', 'city_ Boston', 'city_ Dallas', 'city_ Los Angeles', 'city_ New York City', 'city_ Portland', 'city_ San Francisco', 'city_ Seattle', 'weekday_weekend_Weekend']


c:\Users\Dalbir\Downloads\Trends-Forecasting-Analytics-MLOps-Vertex-AI\venv\Lib\site-packages\sklearn\feature_selection\_rfe.py:300: UserWarning: Found n_features_to_select=30 > n_features=23. There will be no feature selection and all features will be kept.
  warnings.warn(
